In [ ]:
%run setup.py

# 03 · Aprendizaje supervisado — Regresión Lineal

**Objetivo:** estimar `Price` a partir de habitaciones, superficie, antigüedad y
ubicación. Se justifica la elección porque el precio es una variable continua.

**Métrica:** MAE, MSE, RMSE y R².

In [ ]:
# CELDA 3: Preparación de variables
df = load_processed()

columnas_region = [c for c in df.columns if c.startswith("Regionname_")]
features = ["Rooms", "BuildingArea", "Antiguedad"] + columnas_region
X = df[features]
y = df["Price"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Datos listos:", X_scaled.shape)

In [ ]:
# Separación 80% entrenamiento / 20% prueba
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)
print(f"Entrenamiento: {X_train.shape[0]:,}  |  Prueba: {X_test.shape[0]:,}")

modelo = LinearRegression()
modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)

## 3.0 Fundamento teórico

La regresión lineal múltiple modela el precio como:

`y = β0 + β1·x1 + β2·x2 + ... + βp·xp + ε`

Los coeficientes se estiman por **mínimos cuadrados ordinarios (MCO)**,
minimizando la suma de errores al cuadrado:

`J(β) = Σ (yi − ŷi)²`

Supuestos principales: linealidad, independencia de errores, homocedasticidad
y normalidad de los residuos. Las métricas usadas son:

- `MAE = (1/n) Σ |yi − ŷi|`
- `RMSE = sqrt((1/n) Σ (yi − ŷi)²)`
- `R² = 1 − SS_res / SS_tot`

## 3.1 Coeficientes del modelo

In [ ]:
coeficientes = pd.DataFrame({
    "variable": features,
    "coeficiente": modelo.coef_,
}).sort_values("coeficiente", ascending=False)

display(coeficientes)
print(f"Intercepto: {modelo.intercept_:,.2f}")
print("\nInterpretación: un coeficiente positivo indica que, al aumentar la")
print("variable (en unidades estandarizadas), el precio tiende a subir.")

## 3.2 Evaluación con MAE, MSE, RMSE y R²

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

metricas = pd.DataFrame({
    "Métrica": ["MAE", "MSE", "RMSE", "R²"],
    "Valor": [mae, mse, rmse, r2],
})
display(metricas)

print(f"Error absoluto medio: ${mae:,.2f}")
print(f"Raíz del error cuadrático medio: ${rmse:,.2f}")
print(f"R²: {r2:.4f}  → el modelo explica el {r2*100:.2f}% de la varianza.")

## 3.3 Análisis de residuos

In [ ]:
residuos = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].scatter(y_pred, residuos, alpha=0.3, s=10)
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_title("Residuos vs predicciones")
axes[0].set_xlabel("Precio predicho")
axes[0].set_ylabel("Residuo (real - predicho)")

axes[1].scatter(y_test, y_pred, alpha=0.3, s=10)
limites = [y_test.min(), y_test.max()]
axes[1].plot(limites, limites, color="red", linestyle="--")
axes[1].set_title("Valor real vs valor predicho")
axes[1].set_xlabel("Precio real")
axes[1].set_ylabel("Precio predicho")

plt.tight_layout()
plt.show()

print(f"Residuo medio: {residuos.mean():,.2f}")
print("Un R² cercano a 0.4 y residuos dispersos indican que el modelo captura")
print("parte de la tendencia, pero omite factores relevantes del precio.")